# Discovery + Silver: `university.professors`

Mismo patron que `01_discovery_silver_students.ipynb`: explorar `bronze.university__professors`, limpiar con pandas, escribir en `silver.university__professors`.

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.university__professors", engine)
df.shape

(200, 9)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

professor_id            object
first_name              object
last_name               object
email                   object
department              object
hired_at                object
_source_file            object
_ingested_at    datetime64[ns]
_dag_run_id             object
dtype: object


,professor_id,first_name,last_name,email,department,hired_at,_source_file,_ingested_at,_dag_run_id
0,PRF-00001,Matias,Sandoval,matias.sandoval4330@lake.local,cs,2010-11-10,university/professors.csv,2026-07-17 15:16:32.589583,manual__2026-07-17T15:16:30+00:00
1,PRF-00002,Gabriel,Arancibia,gabriel.arancibia7939@demo.io,cs,2021-05-27,university/professors.csv,2026-07-17 15:16:32.589583,manual__2026-07-17T15:16:30+00:00
2,PRF-00003,Rodrigo,Sandoval,rodrigo.sandoval265@demo.io,cs,2022-06-20,university/professors.csv,2026-07-17 15:16:32.589583,manual__2026-07-17T15:16:30+00:00
3,PRF-00004,Matias,Fuentes,matias.fuentes5187@mail.test,math,2009-09-01,university/professors.csv,2026-07-17 15:16:32.589583,manual__2026-07-17T15:16:30+00:00
4,PRF-00005,Lucia,Gutierrez,lucia.gutierrez6256@demo.io,cs,2012-03-23,university/professors.csv,2026-07-17 15:16:32.589583,manual__2026-07-17T15:16:30+00:00


## 2. Nulos y duplicados

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("professor_id duplicados:", df["professor_id"].duplicated().sum())
print("Filas 100% duplicadas:", df.duplicated().sum())

Nulos por columna:
professor_id    0
first_name      0
last_name       0
email           0
department      0
hired_at        0
_source_file    0
_ingested_at    0
_dag_run_id     0
dtype: int64

professor_id duplicados: 0
Filas 100% duplicadas: 0


## 3. Rangos y valores raros

- `department`: cuantos valores distintos hay (esperamos categorias limpias tipo `cs`, `math`, etc).
- `hired_at`: que sea una fecha valida y no este en el futuro.

In [4]:
print("Valores distintos de department:")
print(df["department"].value_counts())
print()

hired = pd.to_datetime(df["hired_at"])
print("Fechas invalidas (no parsean):", hired.isna().sum())
print("hired_at en el futuro:", (hired > pd.Timestamp.now()).sum())

Valores distintos de department:
department
cs            53
math          31
economics     31
history       20
physics       17
biology       17
literature    17
chemistry     14
Name: count, dtype: int64

Fechas invalidas (no parsean): 0
hired_at en el futuro: 0


## 4. Conclusion: reglas de limpieza

Tabla limpia (sin nulos, sin duplicados, `department` con 8 valores consistentes en `snake_case`). Mismas reglas de tipado/estandarizacion que `students`, sin necesidad de descartar filas:

- `first_name`, `last_name` -> `strip()`.
- `email` -> `strip()` + minusculas.
- `hired_at` -> castear de texto a fecha real.
- `department` -> `strip()` + minusculas (ya vienen asi, se fuerza el estandar de todas formas).
- `professor_id` ya es unico, queda como esta (PK en silver).

## 5. Limpieza con pandas

In [5]:
df_silver = df[["professor_id", "first_name", "last_name", "email", "department", "hired_at"]].copy()

df_silver["first_name"] = df_silver["first_name"].str.strip()
df_silver["last_name"] = df_silver["last_name"].str.strip()
df_silver["email"] = df_silver["email"].str.strip().str.lower()
df_silver["department"] = df_silver["department"].str.strip().str.lower()
df_silver["hired_at"] = pd.to_datetime(df_silver["hired_at"]).dt.date

df_silver.head()

,professor_id,first_name,last_name,email,department,hired_at
0,PRF-00001,Matias,Sandoval,matias.sandoval4330@lake.local,cs,2010-11-10
1,PRF-00002,Gabriel,Arancibia,gabriel.arancibia7939@demo.io,cs,2021-05-27
2,PRF-00003,Rodrigo,Sandoval,rodrigo.sandoval265@demo.io,cs,2022-06-20
3,PRF-00004,Matias,Fuentes,matias.fuentes5187@mail.test,math,2009-09-01
4,PRF-00005,Lucia,Gutierrez,lucia.gutierrez6256@demo.io,cs,2012-03-23


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df), "se perdieron o duplicaron filas en la limpieza"
assert df_silver.isna().sum().sum() == 0, "aparecieron nulos nuevos"
assert df_silver["professor_id"].is_unique, "professor_id ya no es unico"
print("OK:", len(df_silver), "filas listas para silver")

OK: 200 filas listas para silver


## 7. Escribir en `silver.university__professors`

`if_exists="replace"` -> full-refresh idempotente, mismo criterio que `students`.

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "university__professors",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)
print("Escrito en silver.university__professors")

Escrito en silver.university__professors


## 8. Verificar lo que quedo en Postgres

In [8]:
check = pd.read_sql("SELECT * FROM silver.university__professors LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT professor_id) AS ids_unicos FROM silver.university__professors", engine))
check

   filas  ids_unicos
0    200         200


,professor_id,first_name,last_name,email,department,hired_at,_silver_loaded_at
0,PRF-00001,Matias,Sandoval,matias.sandoval4330@lake.local,cs,2010-11-10,2026-07-17 15:16:59.939443+00:00
1,PRF-00002,Gabriel,Arancibia,gabriel.arancibia7939@demo.io,cs,2021-05-27,2026-07-17 15:16:59.939443+00:00
2,PRF-00003,Rodrigo,Sandoval,rodrigo.sandoval265@demo.io,cs,2022-06-20,2026-07-17 15:16:59.939443+00:00
3,PRF-00004,Matias,Fuentes,matias.fuentes5187@mail.test,math,2009-09-01,2026-07-17 15:16:59.939443+00:00
4,PRF-00005,Lucia,Gutierrez,lucia.gutierrez6256@demo.io,cs,2012-03-23,2026-07-17 15:16:59.939443+00:00
